In [1]:
import pandas as pd

core = pd.read_csv(
    "../Data/processed/transfermarkt_merged/transfer_core.csv"
)

fbref = pd.read_csv(
    "../Data/processed/fbref_2017_2024_clean/fbref_player_squad_season_clean.csv"
)

print("Transfer core:", core.shape)
print("FBref:", fbref.shape)

Transfer core: (35139, 63)
FBref: (18243, 81)


In [2]:
core.columns.tolist()

['source_file',
 'source_row_number',
 'player_id',
 'transfer_date',
 'transfer_season',
 'from_club_id',
 'to_club_id',
 'from_club_name',
 'to_club_name',
 'transfer_fee',
 'market_value_in_eur',
 'player_name',
 'player_name_normalized',
 'from_club_name_normalized',
 'to_club_name_normalized',
 'transfer_season_derived',
 'transfer_season_matches_date',
 'transfer_fee_status',
 'has_reported_market_value',
 'transfer_event_id',
 'player_canonical_name',
 'player_canonical_name_normalized',
 'player_canonical_name_source',
 'player_present_in_snapshot',
 'player_first_observed_date',
 'player_last_observed_date',
 'player_observation_count',
 'player_name_variant_count',
 'player_source_table_count',
 'player_date_of_birth',
 'player_birth_year',
 'player_country_of_birth',
 'player_country_of_citizenship',
 'player_snapshot_position',
 'player_snapshot_sub_position',
 'player_preferred_foot',
 'player_height_in_cm',
 'player_code',
 'player_snapshot_url',
 'from_club_canonical_nam

In [3]:
fbref.columns.tolist()

['fbref_row_id',
 'player_squad_season_key',
 'player_identity_key',
 'source_file',
 'source_row_number',
 'source_rank',
 'player_name',
 'player_name_normalized',
 'nationality',
 'nationality_raw',
 'position',
 'is_goalkeeper',
 'squad_name',
 'squad_name_normalized',
 'competition_name',
 'competition_id',
 'season',
 'season_start_year',
 'season_end_year',
 'season_end_date',
 'age',
 'birth_year',
 'age_birth_year_consistent',
 'matches_played',
 'total_minutes',
 'minutes_per_appearance',
 'nineties',
 'goals',
 'assists',
 'goals_and_assists',
 'non_penalty_goals',
 'penalty_kicks_made',
 'expected_goals',
 'non_penalty_expected_goals',
 'progressive_carries',
 'progressive_passes',
 'goals_per90',
 'assists_per90',
 'tackles_attempted',
 'tackles_won',
 'dribbles_tackled_pct',
 'shots_blocked',
 'passes_blocked',
 'interceptions',
 'clearances',
 'errors_made',
 'goals_against',
 'goals_against_per90',
 'saves',
 'saves_pct',
 'clean_sheets',
 'clean_sheets_pct',
 'penalty_

In [4]:
needed_core = [
    "player_canonical_name_normalized",
    "player_birth_year",
    "from_club_canonical_name_normalized",
    "to_club_canonical_name_normalized",
    "transfer_date",
    "transfer_event_id"
]

needed_fbref = [
    "player_name_normalized",
    "birth_year",
    "squad_name_normalized",
    "season_start_year",
    "season",
    "total_minutes",
    "goals",
    "assists"
]

print("Core columns:")
for col in needed_core:
    print(col, "✅" if col in core.columns else "❌")

print("\nFBref columns:")
for col in needed_fbref:
    print(col, "✅" if col in fbref.columns else "❌")

Core columns:
player_canonical_name_normalized ✅
player_birth_year ✅
from_club_canonical_name_normalized ✅
to_club_canonical_name_normalized ✅
transfer_date ✅
transfer_event_id ✅

FBref columns:
player_name_normalized ✅
birth_year ✅
squad_name_normalized ✅
season_start_year ✅
season ✅
total_minutes ✅
goals ✅
assists ✅


In [5]:
# Convert dates and years to usable numeric formats

core["transfer_date"] = pd.to_datetime(
    core["transfer_date"],
    errors="coerce"
)

core["transfer_season_start_year"] = (
    core["transfer_date"].dt.year
    - (core["transfer_date"].dt.month < 7).astype(int)
)

core["player_birth_year"] = pd.to_numeric(
    core["player_birth_year"],
    errors="coerce"
)

fbref["birth_year"] = pd.to_numeric(
    fbref["birth_year"],
    errors="coerce"
)

fbref["season_start_year"] = pd.to_numeric(
    fbref["season_start_year"],
    errors="coerce"
)

print("Invalid transfer dates:", core["transfer_date"].isna().sum())
print("Missing core birth years:", core["player_birth_year"].isna().sum())
print("Missing FBref birth years:", fbref["birth_year"].isna().sum())

Invalid transfer dates: 0
Missing core birth years: 27
Missing FBref birth years: 0


In [6]:
core_player_keys = set(
    core[
        [
            "player_canonical_name_normalized",
            "player_birth_year"
        ]
    ]
    .dropna()
    .itertuples(index=False, name=None)
)

fbref_player_keys = set(
    fbref[
        [
            "player_name_normalized",
            "birth_year"
        ]
    ]
    .dropna()
    .itertuples(index=False, name=None)
)

common_player_keys = core_player_keys.intersection(
    fbref_player_keys
)

print("Unique Transfermarkt players:", len(core_player_keys))
print("Unique FBref players:", len(fbref_player_keys))
print(
    "Players matched by name and birth year:",
    len(common_player_keys)
)

Unique Transfermarkt players: 4339
Unique FBref players: 5967
Players matched by name and birth year: 777


In [7]:
matched_player_keys = pd.DataFrame(
    list(common_player_keys),
    columns=[
        "player_canonical_name_normalized",
        "player_birth_year"
    ]
)

core_matchable = core.merge(
    matched_player_keys,
    on=[
        "player_canonical_name_normalized",
        "player_birth_year"
    ],
    how="inner"
)

core_matchable = core_matchable[
    core_matchable["transfer_season_start_year"].between(2017, 2023)
].copy()

print("Matchable transfer rows:", len(core_matchable))
print(
    "Unique matched players:",
    core_matchable["player_id"].nunique()
)

Matchable transfer rows: 3783
Unique matched players: 746


In [8]:
fbref_pre = fbref.add_suffix("_pre")

pre_merged = core_matchable.merge(
    fbref_pre,
    left_on=[
        "player_canonical_name_normalized",
        "player_birth_year",
        "from_club_name_normalized",
        "transfer_season_start_year"
    ],
    right_on=[
        "player_name_normalized_pre",
        "birth_year_pre",
        "squad_name_normalized_pre",
        "season_start_year_pre"
    ],
    how="left",
    validate="many_to_one"
)

pre_matches = pre_merged["fbref_row_id_pre"].notna().sum()

print("Transfer rows:", len(pre_merged))
print("Pre-transfer matches:", pre_matches)
print(
    "Pre-transfer match rate:",
    round(pre_matches / len(pre_merged) * 100, 2),
    "%"
)

Transfer rows: 3783
Pre-transfer matches: 287
Pre-transfer match rate: 7.59 %


In [9]:
season_candidates = core_matchable.merge(
    fbref_pre,
    left_on=[
        "player_canonical_name_normalized",
        "player_birth_year",
        "transfer_season_start_year"
    ],
    right_on=[
        "player_name_normalized_pre",
        "birth_year_pre",
        "season_start_year_pre"
    ],
    how="left"
)

candidate_events = (
    season_candidates.loc[
        season_candidates["fbref_row_id_pre"].notna(),
        "transfer_event_id"
    ]
    .nunique()
)

print(
    "Transfers with an FBref player-season available:",
    candidate_events
)

print(
    "Potential player-season coverage:",
    round(candidate_events / len(core_matchable) * 100, 2),
    "%"
)

Transfers with an FBref player-season available: 1619
Potential player-season coverage: 42.8 %


In [10]:
candidate_counts = (
    season_candidates[
        season_candidates["fbref_row_id_pre"].notna()
    ]
    .groupby("transfer_event_id")
    .size()
)

print(candidate_counts.value_counts().sort_index())

1    1424
2     195
Name: count, dtype: int64


In [11]:
# Start with the exact club matches
exact_matches = pre_merged[
    pre_merged["fbref_row_id_pre"].notna()
].copy()

exact_event_ids = set(exact_matches["transfer_event_id"])

# Keep player-season candidates for events not already matched exactly
fallback_candidates = season_candidates[
    season_candidates["fbref_row_id_pre"].notna()
    & ~season_candidates["transfer_event_id"].isin(exact_event_ids)
].copy()

# Count FBref candidates per transfer event
fallback_candidates["candidate_count"] = (
    fallback_candidates.groupby("transfer_event_id")["fbref_row_id_pre"]
    .transform("count")
)

# Only accept events with exactly one possible FBref row
unique_fallback = fallback_candidates[
    fallback_candidates["candidate_count"] == 1
].copy()

print("Exact club matches:", exact_matches["transfer_event_id"].nunique())
print("Unique fallback matches:", unique_fallback["transfer_event_id"].nunique())

Exact club matches: 287
Unique fallback matches: 1230


In [12]:
pre_final = pd.concat(
    [exact_matches, unique_fallback],
    ignore_index=True
)

# Keep one row per transfer event
pre_final = pre_final.drop_duplicates(
    subset="transfer_event_id"
)

print("Final pre-transfer matches:", len(pre_final))
print(
    "Final pre-transfer coverage:",
    round(len(pre_final) / len(core_matchable) * 100, 2),
    "%"
)

print(
    "Duplicate transfer events:",
    pre_final["transfer_event_id"].duplicated().sum()
)

Final pre-transfer matches: 1517
Final pre-transfer coverage: 40.1 %
Duplicate transfer events: 0


In [13]:
# Convert transfer_date to datetime (safe even if already converted)
core_matchable["transfer_date"] = pd.to_datetime(core_matchable["transfer_date"])

# Determine which FBref season should be considered the player's first season
# at the destination club.
# Jan–May transfers -> same season
# Jun–Aug transfers -> next season

core_matchable["post_season_start_year"] = (
    core_matchable["transfer_season_start_year"]
    + core_matchable["transfer_date"].dt.month.between(6, 8).astype(int)
)

In [15]:
fbref_post = fbref.add_suffix("_post")

In [16]:
post_candidates = core_matchable.merge(
    fbref_post,
    left_on=[
        "player_canonical_name_normalized",
        "player_birth_year",
        "post_season_start_year"
    ],
    right_on=[
        "player_name_normalized_post",
        "birth_year_post",
        "season_start_year_post"
    ],
    how="left"
)

post_candidate_events = (
    post_candidates.loc[
        post_candidates["fbref_row_id_post"].notna(),
        "transfer_event_id"
    ]
    .nunique()
)

print(
    "Transfers with post-transfer FBref season available:",
    post_candidate_events
)

print(
    "Potential post-transfer coverage:",
    round(post_candidate_events / len(core_matchable) * 100, 2),
    "%"
)

Transfers with post-transfer FBref season available: 1471
Potential post-transfer coverage: 38.88 %


In [17]:
# Exact destination-club matches
post_exact = core_matchable.merge(
    fbref_post,
    left_on=[
        "player_canonical_name_normalized",
        "player_birth_year",
        "to_club_name_normalized",
        "post_season_start_year"
    ],
    right_on=[
        "player_name_normalized_post",
        "birth_year_post",
        "squad_name_normalized_post",
        "season_start_year_post"
    ],
    how="left",
    validate="many_to_one"
)

exact_post = post_exact[
    post_exact["fbref_row_id_post"].notna()
].copy()

exact_post_ids = set(exact_post["transfer_event_id"])

# Fallback candidates not already matched exactly
fallback_post = post_candidates[
    post_candidates["fbref_row_id_post"].notna()
    & ~post_candidates["transfer_event_id"].isin(exact_post_ids)
].copy()

fallback_post["candidate_count"] = (
    fallback_post.groupby("transfer_event_id")["fbref_row_id_post"]
    .transform("count")
)

unique_post_fallback = fallback_post[
    fallback_post["candidate_count"] == 1
].copy()

# Combine exact and unique fallback matches
post_final = pd.concat(
    [exact_post, unique_post_fallback],
    ignore_index=True
)

post_final = post_final.drop_duplicates(
    subset="transfer_event_id"
)

print("Exact post matches:", exact_post["transfer_event_id"].nunique())
print(
    "Unique post fallback matches:",
    unique_post_fallback["transfer_event_id"].nunique()
)
print("Final post-transfer matches:", len(post_final))
print(
    "Final post-transfer coverage:",
    round(len(post_final) / len(core_matchable) * 100, 2),
    "%"
)
print(
    "Duplicate transfer events:",
    post_final["transfer_event_id"].duplicated().sum()
)

Exact post matches: 422
Unique post fallback matches: 960
Final post-transfer matches: 1382
Final post-transfer coverage: 36.53 %
Duplicate transfer events: 0


In [18]:
complete_event_ids = set(pre_final["transfer_event_id"]).intersection(
    set(post_final["transfer_event_id"])
)

print(
    "Transfers with both pre and post performance:",
    len(complete_event_ids)
)

Transfers with both pre and post performance: 1025


In [19]:
complete_event_ids = set(pre_final["transfer_event_id"]).intersection(
    set(post_final["transfer_event_id"])
)

print(
    "Transfers with both pre and post performance:",
    len(complete_event_ids)
)

Transfers with both pre and post performance: 1025


In [20]:
pre_complete = pre_final[
    pre_final["transfer_event_id"].isin(complete_event_ids)
].copy()

post_complete = post_final[
    post_final["transfer_event_id"].isin(complete_event_ids)
].copy()

post_stats = post_complete[
    [
        col for col in post_complete.columns
        if col.endswith("_post")
    ] + ["transfer_event_id"]
].copy()

model_dataset = pre_complete.merge(
    post_stats,
    on="transfer_event_id",
    how="inner",
    validate="one_to_one"
)

print("Final modeling dataset shape:", model_dataset.shape)
print(
    "Duplicate transfer events:",
    model_dataset["transfer_event_id"].duplicated().sum()
)

Final modeling dataset shape: (1025, 227)
Duplicate transfer events: 0


In [21]:
model_dataset.to_csv(
    "../Data/processed/model_dataset.csv",
    index=False
)

print("Saved model_dataset.csv")

Saved model_dataset.csv


In [22]:
print("Shape:", model_dataset.shape)
print("Duplicate transfers:", model_dataset["transfer_event_id"].duplicated().sum())

missing_pct = (
    model_dataset.isna().mean() * 100
).sort_values(ascending=False)

missing_pct.head(20)

Shape: (1025, 227)
Duplicate transfers: 0


saves_pct_pre                95.219512
crosses_stopped_pct_pre      95.219512
penalty_saves_pct_pre        95.219512
clean_sheets_pct_pre         95.219512
clean_sheets_pre             95.219512
saves_pre                    95.219512
goals_against_per90_pre      95.219512
goals_against_pre            95.219512
penalty_saves_pct_post       94.439024
goals_against_post           94.439024
goals_against_per90_post     94.439024
saves_post                   94.439024
saves_pct_post               94.439024
clean_sheets_post            94.439024
clean_sheets_pct_post        94.439024
crosses_stopped_pct_post     94.439024
candidate_count              18.731707
dribbles_tackled_pct_pre     13.756098
transfer_fee                 10.341463
dribbles_tackled_pct_post     3.609756
dtype: float64

In [23]:
injuries = pd.read_csv(
    "/Users/davidnavarrete/Downloads/football-datasets-main/datalake/transfermarkt/player_injuries/player_injuries.csv"
)

master_ids = set(model_dataset["player_id"].dropna().astype(int))
injury_ids = set(injuries["player_id"].dropna().astype(int))

matched_ids = master_ids & injury_ids

print("Master unique players:", len(master_ids))
print("Injury unique players:", len(injury_ids))
print("Matched players:", len(matched_ids))
print("Match rate:", round(len(matched_ids) / len(master_ids) * 100, 2), "%")

Master unique players: 488
Injury unique players: 34561
Matched players: 448
Match rate: 91.8 %
